# 🤖 Prática 4: IA e Agentes
### Workshop Dados na Prática · Nerd Data Lab + Programa AI

Até aqui, olhamos para **números**. Mas uma parte enorme dos dados de uma empresa é **texto**: avaliações, e-mails, mensagens de WhatsApp, chamados.

O Café Tambaú tem quase 1.700 avaliações escritas pelos clientes. Ninguém tem tempo de ler todas. Vamos ver três jeitos de entender o que elas dizem:

1. **O jeito antigo:** regras com palavras-chave
2. **Com IA, sem programar:** usando um chatbot (ChatGPT, Gemini ou Claude)
3. **Com IA no código:** classificação automática e um agente que responde perguntas sobre os dados

**Como usar:** Shift + Enter para rodar. Nas células com ✏️, troque os `___`.

In [ ]:
# ⚙️ Configuração: rode esta célula primeiro (Shift + Enter)
import os
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (10, 4)

# De onde vêm os dados:
# 1) Se você subiu a pasta cafe_tambau para o Colab (ícone de pasta à esquerda), ela é usada.
# 2) Se não, os arquivos são baixados deste endereço.
URL_BASE = "https://raw.githubusercontent.com/wuldson-franco/workshop_dados_pratica/main/"
PASTA_LOCAL = "./"

def ler(arquivo, **opcoes):
    caminho = PASTA_LOCAL + arquivo
    if os.path.exists(caminho):
        return pd.read_csv(caminho, **opcoes)
    return pd.read_csv(URL_BASE + arquivo, **opcoes)

print("Tudo pronto! ✅")

import duckdb

In [ ]:
avaliacoes = ler("dados_tratados/avaliacoes.csv", parse_dates=["data_avaliacao"])
avaliacoes["mes"] = avaliacoes["data_avaliacao"].dt.month
avaliacoes = avaliacoes[avaliacoes["mes"] <= 9]
avaliacoes[["data_avaliacao", "canal", "nota", "comentario"]].sample(10, random_state=3)

## 1. O jeito antigo: palavras-chave

Criamos uma lista de palavras para cada tema e marcamos a avaliação se ela contém alguma dessas palavras.

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
temas = {
    "Preço": ["caro", "preço", "12,90", "compensa"],
    "Delivery": ["ifood", "entrega", "delivery", "frio", "demor", "atrasad"],
    "Concorrência": ["café novo", "concorrente", "___"],
}

def classificar_por_palavras(texto):
    texto = texto.lower()
    for tema, palavras in temas.items():
        if any(palavra in texto for palavra in palavras):
            return tema
    return "Outros"

avaliacoes["tema_palavras"] = avaliacoes["comentario"].apply(classificar_por_palavras)
avaliacoes["tema_palavras"].value_counts()

In [ ]:
avaliacoes[avaliacoes["tema_palavras"] == "Preço"]["comentario"].value_counts()

In [ ]:
reclamacoes = avaliacoes[avaliacoes["tema_palavras"] != "Outros"]
pd.crosstab(reclamacoes["mes"], reclamacoes["tema_palavras"]).plot(
    kind="bar", stacked=True, title="Reclamações por tema e mês", color=["#3DF29A", "#1FD3F0", "#f5a524"])
plt.xticks(rotation=0)
plt.show()

🤔 **Funciona, mas tem limites.** Veja alguns problemas:

- Olhe o tema "Preço": ele pegou "Preço justo e comida gostosa", que é um **elogio**! A palavra estava lá, mas o sentido era o oposto.
- E se alguém escrever "a espera foi enorme"? Nenhuma palavra da lista aparece, e a reclamação passa despercebida.
- Um comentário pode falar de **dois temas** ao mesmo tempo.
- Para cada tema novo, alguém precisa pensar em todas as palavras possíveis.

A IA resolve isso porque ela **entende o sentido** do texto, e não só as palavras.

## 2. Com IA, sem programar ✨

Rode a célula abaixo. Ela monta um prompt com 30 avaliações dos últimos meses. **Copie o texto gerado**, cole no ChatGPT, Gemini ou Claude, e veja o que a IA responde.

In [ ]:
amostra = avaliacoes[avaliacoes["mes"] >= 7].sample(30, random_state=10)

prompt = """Você é analista de dados de uma cafeteria em João Pessoa.
Classifique cada avaliação abaixo em UM tema: Preço, Delivery, Concorrência, Atendimento, Produto ou Elogio.
Também diga o sentimento: Positivo, Neutro ou Negativo.

Responda SOMENTE em formato CSV, sem explicações, com o cabeçalho:
id;tema;sentimento

Avaliações:
"""
for linha in amostra.itertuples():
    prompt += f"{linha.id_avaliacao}: {linha.comentario}\n"

prompt += "\nDepois do CSV, escreva um parágrafo com as 3 principais recomendações para o dono do café."
print(prompt)

Agora **cole a resposta da IA** (só a parte do CSV) entre as aspas abaixo e rode a célula.

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
resposta_da_ia = """
___
"""

from io import StringIO
linhas = resposta_da_ia.strip()
if linhas and linhas != "___":
    resultado = pd.read_csv(StringIO(linhas), sep=";")
    print(resultado["tema"].value_counts())
    resultado["tema"].value_counts().plot(kind="barh", color="#1FD3F0", title="Temas segundo a IA")
    plt.show()
else:
    print("Cole a resposta da IA entre as aspas e rode de novo.")

🤔 **Para pensar:** a IA concordou com as palavras-chave? Ela pegou alguma reclamação que as palavras-chave deixaram passar? E as recomendações que ela deu fazem sentido com o que você viu nas Práticas 2 e 3?

⚠️ **Cuidado:** nunca cole dados reais de clientes (nomes, CPFs, telefones) em chatbots públicos. Aqui pode, porque os dados são fictícios.

## 3. Com IA no código: um agente que consulta os dados 🧑‍💻

Esta parte usa a API do **Gemini**, que tem uma camada gratuita. Para rodar:

1. Crie uma chave em **aistudio.google.com** ("Get API key").
2. No Colab, clique no ícone de 🔑 (Secrets) à esquerda, adicione `GEMINI_API_KEY` com a sua chave e ative o acesso do notebook.

Se não tiver chave, tudo bem: acompanhe a demonstração no telão.

In [ ]:
!pip install -q google-genai

GEMINI_API_KEY = None
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

cliente_ia = None
if GEMINI_API_KEY:
    from google import genai
    cliente_ia = genai.Client(api_key=GEMINI_API_KEY)
    print("IA conectada! ✅")
else:
    print("Sem chave do Gemini: as próximas células vão só mostrar o que fariam.")

# Use um modelo disponível no AI Studio. Se este nome der erro, troque pelo modelo "flash" mais recente.
MODELO = "gemini-2.5-flash"

def perguntar_ia(texto):
    if cliente_ia is None:
        return None
    resposta = cliente_ia.models.generate_content(model=MODELO, contents=texto)
    return resposta.text

### 3.1 O agente analista

Um **agente** é uma IA que não só responde, mas **age**: aqui, ele recebe uma pergunta em português, **escreve o SQL sozinho**, executa nos dados e explica o resultado.

```
Pergunta  →  IA escreve o SQL  →  o código executa  →  IA explica a resposta
```

In [ ]:
vendas = ler("dados_tratados/vendas_limpas.csv", parse_dates=["data_hora"])

esquema = """
Tabela vendas (uma linha por item vendido):
id_venda, data_hora, data, hora, periodo (Manhã, Almoço, Tarde, Noite), mes (1 a 9), trimestre (T1, T2, T3),
id_cliente, id_produto, nome_produto, categoria, quantidade, preco_unitario, valor_total,
custo_unitario, custo_total, lucro, canal (Balcão ou iFood), forma_pagamento, tempo_entrega_min
"""

def agente(pergunta):
    print("❓ Pergunta:", pergunta)
    sql_gerado = perguntar_ia(
        f"Você escreve SQL para DuckDB. {esquema}\n"
        f"Escreva UMA consulta SQL que responda: {pergunta}\n"
        "Responda apenas com o SQL, sem explicações e sem ```."
    )
    if sql_gerado is None:
        print("(Sem chave do Gemini: aqui a IA escreveria o SQL, o código rodaria e a IA explicaria o resultado.)")
        return
    sql_gerado = sql_gerado.replace("```sql", "").replace("```", "").strip()
    print("\n🛠️ SQL que a IA escreveu:\n", sql_gerado)

    resultado = duckdb.sql(sql_gerado).df()
    print("\n📊 Resultado:")
    display(resultado.head(15))

    explicacao = perguntar_ia(
        f"Pergunta do dono da cafeteria: {pergunta}\n"
        f"Resultado da consulta:\n{resultado.head(15).to_string()}\n"
        "Explique o resultado em 2 frases simples, em português, e sugira uma ação."
    )
    print("\n💬 Explicação da IA:\n", explicacao)

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
agente("Qual foi o faturamento do iFood em cada mês e qual o tempo médio de entrega?")

Agora é a sua vez: faça a sua própria pergunta ao agente!

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
agente("___")

🤔 **Para pensar:** a IA sempre acerta o SQL? O que aconteceria se ela errasse e ninguém conferisse? É por isso que, em 2026, saber **SQL e dados continua essencial**: é você quem garante que a resposta do agente está certa.

## 🎯 Fechando o case: o que descobrimos hoje

| Prática | O que descobrimos |
|---|---|
| 🛠️ Engenharia | |
| 📊 Análise | |
| 🧠 Ciência | |
| 🤖 IA | |

**As 3 causas da queda nas vendas:**
1. ✍️
2. ✍️
3. ✍️

**As 3 recomendações para o dono do Café Tambaú:**
1. ✍️
2. ✍️
3. ✍️

📋 Envie suas recomendações no formulário para ganhar seu cupom do sorteio! 🎁